In [ ]:
import os
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

print("INITIALIZING SMART RENTAL TRACKING ENGINE...")
print("-" * 60)

db_path = 'equipment_rental.db'

# 1. ROBUST CONNECTION & INGESTION
if not os.path.exists(db_path):
    raise FileNotFoundError(f" DB Missing'{db_path}' is in exactly this folder: {os.getcwd()}")

conn = sqlite3.connect(db_path)

# Extracting the exact table defined in the hackathon challenge prompt
try:
    df = pd.read_sql_query("SELECT * FROM EquipmentRental", conn)
    print(f" Loaded {len(df)} records from the database.")
except pd.io.sql.DatabaseError:
    # Fallback in case your table is just named 'rentals' or 'equipment'
    try:
        df = pd.read_sql_query("SELECT * FROM rentals", conn)
        print(" Loaded records from the 'rentals' table.")
    except pd.io.sql.DatabaseError:
        raise ValueError("Could not find the 'EquipmentRental' or 'rentals' table in your database.")
finally:
    conn.close()

# 2. DATA CLEANING & STANDARDIZATION
# The database might contain literal string 'NULL' values which need to be converted to actual NaNs
df.replace(['NULL', 'None', '', ' '], np.nan, inplace=True)

# Convert date columns to datetime objects for accurate chronological calculations
date_cols = ['CheckInDate', 'CheckOutDate', 'ExpectedReturnDate']
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Ensure numeric columns are properly typed
numeric_cols = ['EngineHoursPerDay', 'IdleHoursPerDay', 'RentalDays']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

print("⚙️ Executing Edge-Case Anomaly Detection...")

# 3. ANOMALY DETECTION (Targeting Hackathon Requirements)

# A. Unassigned Equipment (Ghost Rentals / Misuse)
if 'SiteID' in df.columns and 'LastOperatorID' in df.columns:
    df['Flag_Unassigned_Asset'] = df['SiteID'].isna() | df['LastOperatorID'].isna()
else:
    df['Flag_Unassigned_Asset'] = False

# B. Long Idle Hours (Misuse & Fuel Waste)
if 'IdleHoursPerDay' in df.columns and 'EngineHoursPerDay' in df.columns:
    df['Flag_High_Idle_Waste'] = (df['IdleHoursPerDay'] > df['EngineHoursPerDay']) & (df['IdleHoursPerDay'] > 5)

    # C. Zero Utilization (Complete Abandonment)
    df['Flag_Zero_Utilization'] = (df['EngineHoursPerDay'] == 0) & (df['RentalDays'] > 0)
else:
    df['Flag_High_Idle_Waste'] = False
    df['Flag_Zero_Utilization'] = False

# D. Logical Date Errors (Data Corruption)
if 'CheckOutDate' in df.columns and 'CheckInDate' in df.columns:
    df['Flag_Date_Error'] = df['CheckOutDate'] < df['CheckInDate']
else:
    df['Flag_Date_Error'] = False

# 4. RESOLUTION ROUTING & PAYLOAD GENERATION
print("📦 Structuring API Payload...")

# Consolidate all flags to determine if an asset requires action
flag_columns = [col for col in df.columns if col.startswith('Flag_')]
df['Requires_Action'] = df[flag_columns].any(axis=1)


# ---------------------------------------------------------
# E. MACHINE LEARNING: MULTIVARIATE INEFFICIENCY (The "Wow" Factor)
# ---------------------------------------------------------
print("🧠 Running Unsupervised ML (Isolation Forest)...")

# We only want to run ML on records that actually have numeric data
if all(col in df.columns for col in ['EngineHoursPerDay', 'IdleHoursPerDay', 'RentalDays']):

    # Create a feature matrix of the numeric data
    ml_features = ['EngineHoursPerDay', 'IdleHoursPerDay', 'RentalDays']
    X = df[ml_features].copy()

    # Standardize the data so a large number (20 Rental Days) doesn't overpower a small one (1.5 Engine Hours)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Initialize Isolation Forest
    # contamination=0.15 means we expect roughly 15% of the fleet to be acting abnormally
    iso_forest = IsolationForest(contamination=0.15, random_state=42)

    # Predict anomalies (-1 is an anomaly, 1 is normal)
    df['Flag_ML_Anomaly'] = iso_forest.fit_predict(X_scaled) == -1
else:
    df['Flag_ML_Anomaly'] = False

def build_alert_tags(row):
    tags = []
    if row.get('Flag_Unassigned_Asset'):
        tags.append({"severity": "high", "issue": "Unassigned Equipment - Security Risk"})
    if row.get('Flag_High_Idle_Waste'):
        tags.append({"severity": "medium", "issue": f"Long Idle Hours ({row.get('IdleHoursPerDay', 0)} hrs/day)"})
    if row.get('Flag_Zero_Utilization'):
        tags.append({"severity": "high", "issue": "Zero Utilization - Asset Abandoned"})
    if row.get('Flag_Date_Error'):
        tags.append({"severity": "critical", "issue": "Chronological Data Corruption"})

    # Add the ML flag here!
    if row.get('Flag_ML_Anomaly'):
        tags.append({"severity": "info", "issue": "AI Flag: Abnormal Usage Pattern Detected"})

    return tags

df['Alert_Details'] = df.apply(build_alert_tags, axis=1)

# Filter down to only the anomalous records
alerts_df = df[df['Requires_Action'] == True].copy()

# Select columns relevant for the MERN frontend dashboard
export_columns = [
    'EquipmentID', 'equipment_id', 'Type', 'type', 'SiteID', 'site_id',
    'LastOperatorID', 'operator_id', 'EngineHoursPerDay', 'IdleHoursPerDay', 'Alert_Details'
]

# Keep only the columns that actually exist in your specific file
final_columns = [col for col in export_columns if col in alerts_df.columns]
final_payload = alerts_df[final_columns]

print("-" * 60)
print(f"🔥 SCAN COMPLETE: {len(alerts_df)} ANOMALIES DETECTED 🔥")
print("-" * 60)

# Export to a structured JSON file for your Express API
json_filename = 'smart_rental_alerts.json'
final_payload.to_json(json_filename, orient='records', date_format='iso')

print(f"✅ Success! Dashboard payload exported to: {json_filename}")

: 